## **CIND 820 - Big Data Analytics Project**

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split # to apply train-test split
from sklearn.ensemble import RandomForestClassifier # to implement Random Forest modeling
from sklearn.model_selection import cross_val_score # to apply k-fold cross-validation 
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report # to evaluate the model's performance
from sklearn.metrics import ConfusionMatrixDisplay # to display the confusion matrix

# Tree Visualisation
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

**Create a data frame of the Room Occupancy dataset. Display the names of the columns. Check the data types of the attributes. Run a summary statistics of the dataset.**

In [ ]:
# Read the CSV file into a data frame called Occupancy
Occupancy = pd.read_csv('Occupancy_Estimation.csv')

# Display the first five records of the data frame
Occupancy.head()

In [ ]:
# Get the column names
print("Column Names:", Occupancy.columns)

In [ ]:
# Get the data types of each column
print("\nData Types of Each Column:")
print(Occupancy.dtypes)

In [ ]:
# Get the summary statistics
print("Summary Statistics:")
print(Occupancy.describe())

**Split the data into training and testing sets**

In [ ]:
# Split the data into features (X) and target(y)
X = Occupancy[['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp',
               'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light',
               'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound',
               'S5_CO2', 'S5_CO2_Slope', 'S6_PIR', 'S7_PIR']]

y = Occupancy['Room_Occupancy_Count']

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.head()

**Experiment with undersampling of the majority class**

In [ ]:
from collections import Counter

# Summarize class distribution
print(Counter(y_train))

In [ ]:
from imblearn.under_sampling import RandomUnderSampler


# Define undersample strategy
undersample = RandomUnderSampler(random_state=42, sampling_strategy='majority')

In [ ]:
# Fit and apply the transform
X_over, y_over = undersample.fit_resample(X_train, y_train)

In [ ]:
# Summarize class distribution
print(Counter(y_over))

**Use Random Forest for modeling**

In [ ]:
# Create a Random Forest classifier
model = RandomForestClassifier()

**Apply cross validation on the training set**

In [ ]:
# Apply cross-validation on the training set
#scores = cross_val_score(model, X_train, y_train, cv=10, scoring='accuracy')
scores = cross_val_score(model, X_over, y_over, cv=10, scoring='accuracy')

# Evaluation metrics
print(f'Cross-Validation Accuracy Scores on Training Set: {scores}')
print(f'Average Cross-Validation Accuracy on Training Set: {scores.mean()}')

**Train the model on the full training set and evaluate on the test set** 

In [ ]:
# Train the model using the training set
#model.fit(X_train, y_train)
model.fit(X_over, y_over)

In [ ]:
# Predict the response using the test set
y_pred = model.predict(X_test)
print(y_pred[:20])

In [ ]:
# Calculate evaluation metrics
test_accuracy = accuracy_score(y_test, y_pred)          # actual (true) labels vs predicted labels

conf_matrix = confusion_matrix(y_test, y_pred)

print("Test Accuracy:", test_accuracy)

In [ ]:
# Display the confusion matrix
ConfusionMatrixDisplay(confusion_matrix=conf_matrix).plot();

In [ ]:
# Calculate the performance metrics on the test set
class_report = classification_report(y_test, y_pred, digits=3)
print(class_report)

**Visualizing the Results**

In [ ]:
# Export the first three decision trees from the forest

for i in range(3):
    tree = model.estimators_[i]
    dot_data = export_graphviz(tree,
                               feature_names=X_train.columns,  
                               filled=True,  
                               max_depth=2, 
                               impurity=False, 
                               proportion=True)
    graph = graphviz.Source(dot_data)
    display(graph)

**Plot the importance of each feature**

In [ ]:
# Create a series containing feature importances from the model and feature names from the training data
feature_importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

# Plot a simple bar chart
feature_importances.plot.bar();

**Rerun the model with PCA components**

In [ ]:
# Read the CSV file into a data frame called PCA
PCA = pd.read_csv('pca.csv', usecols=range(1,9)) # Read all columns except the first one

# Display the first five records of the data frame
PCA.head()

In [ ]:
# Define the predictor and target variables
X = PCA.iloc[:, 0:7]
y  = PCA['Room_Occupancy_Count']

X.head()

In [ ]:
# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Create a Random Forest classifier
model_pca = RandomForestClassifier()

In [ ]:
# Apply cross-validation on the training set
scores = cross_val_score(model_pca, X_train, y_train, cv=10, scoring='accuracy')

# Evaluation metrics
print(f'Cross-Validation Accuracy Scores on Training Set: {scores}')
print(f'Average Cross-Validation Accuracy on Training Set: {scores.mean()}')

In [ ]:
# Train the model using the training set
model_pca.fit(X_train, y_train)

In [ ]:
# Predict the response using the test set
y_pred = model_pca.predict(X_test)

In [ ]:
# Calculate evaluation metrics
test_accuracy = accuracy_score(y_test, y_pred)          # actual (true) labels vs predicted labels

conf_matrix = confusion_matrix(y_test, y_pred)

print("Test Accuracy:", test_accuracy)

In [ ]:
# Display the confusion matrix
ConfusionMatrixDisplay(confusion_matrix=conf_matrix).plot();

In [ ]:
# Calculate the performance metrics on the test set
class_report = classification_report(y_test, y_pred, digits=3)
print(class_report)